# Fraud / Anomaly Detection
## Notebook 09 — One-Class SVM

### Objective
Learn a nonlinear boundary around the majority transaction pattern.

## Imports and sample

In [ ]:
from pathlib import Path
import sys, numpy as np, pandas as pd
ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from src.fraud_detection import load_features_labels, evaluate_scores, save_results
X, y = load_features_labels(ROOT)
RANDOM_STATE = 42
print('Features:', X.shape, '| fraud:', int(y.sum()))
from src.fraud_detection import stratified_working_sample
from sklearn.svm import OneClassSVM
Xw, yw = stratified_working_sample(X, y, max_rows=15000, random_state=RANDOM_STATE)

**Interpretation.** Kernel SVM training scales poorly with row count, so this notebook uses a documented cap.

**ML decision.** Treat One-Class SVM as a quality-versus-runtime comparison candidate.

## Fit and evaluate

In [ ]:
model = OneClassSVM(kernel='rbf', gamma='scale', nu=max(yw.mean(), 0.005))
model.fit(Xw)
scores = -model.decision_function(Xw).ravel()
result = evaluate_scores(yw, scores, 'One-Class SVM')
pd.DataFrame({'row_index': Xw.index, 'label': yw, 'score': scores}).to_csv(ROOT/'reports/one_class_svm_scores.csv', index=False)
result

**Interpretation.** The decision score measures distance beyond the learned normal boundary.

**ML decision.** Reject this model if its gain does not justify substantially higher runtime.